# 03_chargement_duckdb

In [1]:
import duckdb
import polars as pl
from pathlib import Path
ROOT_PATH = Path.cwd().resolve().parent

In [2]:
# Chargement des dataframes polars
def load_processed(file_name):
    '''
   Renvoie la dataframe polars à partir des données dont le nom (avec son extension) est donné en entrée.
    '''
    path_target_file = ROOT_PATH / "data" / "processed" / file_name
    return pl.read_parquet(path_target_file)

# Chargement des fichiers dans des dataframes polars
poi_df = load_processed("2026-07-22_143436_paris_poi.parquet")
connections_df = load_processed("2026-07-22_143436_connections.parquet")

print(poi_df.shape)
print(connections_df.shape)

(50, 9)
(99, 10)


In [3]:
# Création du connecteur duckdb
con = duckdb.connect(str(ROOT_PATH / "data" / "warehouse" / "electric_mobility.duckdb"))

In [4]:
# Création de la table poi
con.execute("""
    CREATE TABLE IF NOT EXISTS poi (
        poi_id BIGINT PRIMARY KEY,
        title VARCHAR,
        town VARCHAR,
        postcode VARCHAR,
        latitude DOUBLE,
        longitude DOUBLE,
        number_of_points BIGINT,
        usage_cost VARCHAR,
        date_last_confirmed VARCHAR
    )
""")

# Insertion des données dans la table poi
con.execute("""
    INSERT INTO poi (poi_id, title, town, postcode, latitude, longitude, number_of_points, usage_cost, date_last_confirmed)
    SELECT           poi_id, title, town, postcode, latitude, longitude, number_of_points, usage_cost, date_last_confirmed 
    FROM poi_df
    ON CONFLICT (poi_id) DO UPDATE SET
        title = EXCLUDED.title,
        town = EXCLUDED.town,
        postcode = EXCLUDED.postcode,
        latitude = EXCLUDED.latitude,
        longitude = EXCLUDED.longitude,
        number_of_points = EXCLUDED.number_of_points,
        usage_cost = EXCLUDED.usage_cost,
        date_last_confirmed = EXCLUDED.date_last_confirmed
""")

In [5]:
# Création de la table connections
con.execute("""
    CREATE TABLE IF NOT EXISTS connections (
        connection_id BIGINT PRIMARY KEY,
        poi_id BIGINT REFERENCES poi(poi_id),
        power_kw DOUBLE,
        amps BIGINT,
        voltage DOUBLE,
        connection_type VARCHAR,
        current_type VARCHAR,
        is_operational BOOLEAN,
        level_title VARCHAR,
        is_fast_charge_capable BOOLEAN
    )
""")

# Insertion des données dans la table connections
con.execute("""
    INSERT INTO connections (connection_id, poi_id, power_kw, amps, voltage, connection_type, current_type, is_operational, level_title, is_fast_charge_capable)
    SELECT                   connection_id, poi_id, power_kw, amps, voltage, connection_type, current_type, is_operational, level_title, is_fast_charge_capable
    FROM connections_df
    ON CONFLICT (connection_id) DO UPDATE SET
        poi_id = EXCLUDED.poi_id,
        power_kw = EXCLUDED.power_kw,
        amps = EXCLUDED.amps,
        voltage = EXCLUDED.voltage,
        connection_type = EXCLUDED.connection_type,
        current_type = EXCLUDED.current_type,
        is_operational = EXCLUDED.is_operational,
        level_title = EXCLUDED.level_title,
        is_fast_charge_capable = EXCLUDED.is_fast_charge_capable
""")

In [6]:
con.execute("SELECT COUNT(*) FROM poi").fetchall()

[(50,)]

In [7]:
con.execute("SELECT COUNT(*) FROM connections").fetchall()

[(99,)]

In [9]:
# Test de Join
con.execute("""
                SELECT 
                        connection_type,
                        power_kw,
                        town
                FROM connections c
                JOIN poi p ON c.poi_id = p.poi_id
""").pl()

connection_type,power_kw,town
str,f64,str
"""CEE 7/4 - Schuko - Type F""",7.0,null
"""CHAdeMO""",22.0,"""Paris"""
"""Type 2 (Tethered Connector) """,22.0,"""Paris"""
"""Type 2 (Socket Only)""",7.0,null
"""CEE 7/4 - Schuko - Type F""",7.0,null
…,…,…
"""SCAME Type 3C (Schneider-Legra…",22.0,"""Paris"""
"""CEE 7/4 - Schuko - Type F""",22.0,null
"""SCAME Type 3C (Schneider-Legra…",22.0,"""Paris"""


In [11]:
#combien de connecteurs par ville ?
con.execute("""
                SELECT 
                        p.town,
                        COUNT(connection_id)
                FROM connections c
                JOIN poi p ON c.poi_id = p.poi_id
                GROUP BY p.town
""").pl()

town,count(connection_id)
str,i64
"""Paris""",37
null,62


In [12]:
con.execute("""
            SELECT p.poi_id, COUNT(c.connection_id)
            FROM connections c
            JOIN poi p ON c.poi_id = p.poi_id
            WHERE p.town IS NULL
            GROUP BY p.poi_id
""").pl()

poi_id,count(c.connection_id)
i64,i64
198881,4
198731,4
198809,2
198815,2
198885,2
…,…
198728,2
198898,2
198895,2


In [14]:
con.execute("""
                SELECT poi_id, title, town
                FROM poi
                WHERE town IS NULL
""").pl()

poi_id,title,town
i64,str,str
198838,"""Paris | Rue Sainte-Croix de la…",null
198862,"""pompidou""",null
198787,"""Paris | Quai de Béthune 36""",null
198777,"""Paris | Quai de la Tournelle 2…",null
198803,"""Paris | Rue Neuve Saint-Pierre…",null
…,…,…
198871,"""Paris | Rue Saint-Martin 204""",null
198881,"""Paris | Boulevard du Temple 27""",null
198831,"""Paris | Boulevard Richard Leno…",null
